# GC-LSTM-GhostNet - Step 3 graph and sequence smoke test

Build strict contiguous flow windows and directed endpoint graphs after leakage-safe preprocessing.


In [ ]:
from pathlib import Path
import base64
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from kaggle_secrets import UserSecretsClient

PROJECT_DIR = Path("/kaggle/working/Luan-Van-GC-LSTM-GhostNet-CICDDoS2019-v1")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "step3_smoke"
MOUNTED_DATA_DIR = Path("/kaggle/input/datasets/dungnguyen28101991/cicddos2019-parquet")
DOWNLOADED_DATA_DIR = Path("/kaggle/working/cicddos2019-parquet-input")
PROJECT_ARCHIVE_B64 = "UEsDBBQAAAAIAE9wDF24bMKt+gUAACQNAAAJAAAAUkVBRE1FLm1kzVfbbttGEH3nVyxQ5M2UbCdNneZJsB3XiO0EtlGgQAFxxR2RG5O7zF4kK1/fM7uULDtp0bcWsAGJnOuZmTOjn8TFaXl1d39dXrTWhxsKwhpxenlanp3Zu+PDo3dFcd9qL5a2U+QEPoWWRG1NcLbrSAlHg7Mq1kFDER+/UB0g7ZKcokD5jTSqqDvpvV7qWmbhVnoSdpkkvwtjkAP8ra17WHZ2PRG3V+Xs9ortsHyhbFx0VAYnB2XhjR4DGZ88OcK3odO1Dt1G2BjYh6/tQCmu1dGkKO4CDV4cJ3OvRR2dI8PSSGClFf1aFKWo8cjJTn9Dln/Mrq8466Vuosvhs60qRTlfSh3aZeyqZK8anETStezmC2TYaUPVe9jzwQGm6KgcHHlyK20a8Vm6rxHpKo0IV+Q2yUQvjV6SD2IF/yr5YwvrVgfyg6yp9HJJSKqlXopaGmvYn/62E4W94PQiBgTvZT/kUknlhayd9X7n2Nm1aJyNg88xSsAqqqwy16pCDZ1eQXvpbC+8ja4GjhpCHOj4HUZYm9sCANnos0lRw2UCa0GAiwTw1Gb6lNQ0cJLcFY3pAXcKgQYJiIkrR65MKsiUjHTaevHL4avk+eTwFUun16U1qF1PSksjUJaL2eWN0P0Qww6PPblLb7sc1QfEBP/IBp5QjReC19pcy0f4RrwoFXsdUrvXhIj5iQt6iVpn6FIxDoRH6wXRkXyQDR38QItzTho8Ha7XBqXS9T5429p4NDH51Gme8MQA6ucdwUXGvK21UXbNsymDMIQ+ErnKuT5lKrIc/MFY6gNGKQWaQx+Nl8EicxKdXFAHa/KBTK47jyimF1hhGAUKQrJuR7e53RwGHV1ylxvi8rMIVpwhU20y1njSYFhbQaoZc6LU78li0D1kaUhdnqyWnQXuo46xiuBVJTgWsn6Ap8VGGNnncYDf32bl8c9vBRk1WG2CALm05HNX9jxwnidhiyJXJZtm2kAsu1LCfWi35WN2sy54MMYZLWXsgvgomwbNjxJgtgOIoqqqAPYpVDSNaeKGzPHJEYjz3dG01rVS1jONlkOuKYsXxfljmq/dZMqodGLebD1bBXm0xbAJLZ6XGD1XT9ir+LMQoiz5YwnYxfQh6Uy1Qb+PLzE6+PLsNTMpum96FaUpf8f/S8Ytwfpb0i9XR9Nsw09zbNluZsCRCP2U6W2ykX03vu5Rppcy35PhvkYGwJdg0TKRyvHhm5OM0anlV+AB5mpQte/tA6XhSc2zsCjTHhn8mC7+FklutuN5tvkvAP3vEn+d+3S6a90nHP4xudf/t+S277Zc05FpUMKjty9fMKdh3Ec0zploMqeuHa8/LKdRcp5n10/M8K06EFUC6sVDvjueFLLEdrtOvnhrqom4B7m5iMuh83Z0sqczUsE8U0HWyXt+hDn2vXSbrbFbuWauk0olzvF8jxSZkn3AvlHMnqBTT99Rz4RvLRovH2WhaywOIRw62rcCXJ1bgWuRbwnmVRc7bJldd2RUmd+LjONk20bw6fIWBlVbp3aXXL5peO0fvZ2e7Bi0TMSsiHczVnTBqxWBJTYHMWNpx37IIUi2zcuHL0BkhkWGPad5s+dsSnpEgmN+i2gUk96Au4L3/MioyJUW1j5gHTvMeGTsOEIZAloA8iPlCqiTK6pnnQzgP/HKXrdk8hbsbcQWwL0qFxxQxhMhOQIsKEdRfZxdXFydz2efL+f3nz6e31RQxHZs2m1Ed4T7BainFY58WR1bLR22dm06K1Xh42Jc7bmPfNLhLR2JvefKI1XD29HuN8H2WN6v/y3aMN0Hz2ebSa/8OtLDh9npp3wlYjC7fLAkvOHC5OoAiQU18IE/3Dqp93Dw7noh3WXcmwW0a0rFz+swbXu74POUtufdVO3t8W17+PcMx65LtVmiKNHsVFfU6hpmgx1sZ5vNRHyIXSd6/UiqzNchGpRkn0+iNAl7vyS0LwapGam0jxnzFPubp3w7a2HDjmcESKjRnEfeAPmI4ArwtETchgUb4l8Dub96hkXzzZwCgP0lx7dtMpjoqbeY6+IvUEsDBBQAAAAIAE9wDF0Pr4zJ/gQAAJcMAAAQAAAAYXNzdW1wdGlvbnMueWFtbJ1WTW8bNxC991fMrS0gubKTOIkLH4TEcAykqRsL6KEoCGp3tMuCS7IkV7by6/tIrlaRLLd1Tpa5nA++N/NmZAh956KyJlx8RzQlVV/Qr/Pp3e3Hm8V0NjvFIVFlO2cNm3iBnyaqprd9EI23vcvfPctgzQUtWiYnHXuqLQcyNlLNK2WYKuli75myTSDrKfDfPZuKaWl7U0uvOJxkZ6pzskKk63LVeV4jMt0rU9v7QJW3ISjTUHBaxa+sadlHkqsVV5EqLQMOpJaIUNwiEdlr+A229xWLldIsVE1O94FW2lr/w/DF2/v04Sd6OXt7/mM2ljqyNzKqNYcL+iOqjkOUnRONdCIwErJmQjhawimcJQcFnwmxqZ1VJopkJcoz/sxecT/28GeBmEwcSC3kSMgxOs4O6VhLrepsK1YesBW7Y4x4dtbHQK9nJE1Nb2aE0yohG71UJiGaADwkbhdgn52rDDTwKOa0jR5IgubZyfmLHGZ28vrsAP/TXWS7ItsD2WnxUYrjGOAr9cC1eCUGwwkZ4I+TXAxil+Nzcb39fHWsyF1+TAWzTuViE3h2H+UjeK/nN59IBeIHxyYkNFYo7QjQV8qHOFQECjGwBpr7EN5uo4zfk6uEfWZsyg/4nh0OWUzx0J6pBbC43RzgmjGcWqM31HGtpKGDpA8wTbl/G16PqlAFq4citB7M/JcouH6pVWizmMhOmWybRCF6hlagpeM+VO/w5gbGCVnPEQ/lele3tQrRK1TvWKUjKHsRLmcns1PUjkCKqpPR+nB5OptNvobOc2cB8zHA9nwJ2Uc7IdBb/kUOier6uYgmFo6VYIOMRLuBvZNedoxMwhOwOm/XqgY6UFRZejB1cgJa+qpVEY2atBf4WsTv1BdO8hsjsDvQ3JuxYijXWqBOboAvmt2jxW2XGcDTW2D3fTLT3CFfeQR66E3Vhssz4LuUsWpFQODLs1fnE2qTHAIYBiNvAaJ2rSxMzGvZJZoGQA4Y2MdDhHtmCGypdrGl4rkMvJ8v5scYuANSWkv/YbG4fYS8XQb2axRhmRkIlwof6KCDV4p1TRJKSKkuUaBJCeOTs1FBlTEJbT329JaORRaS5C1rTKX7GiEzD3nCKXwELywTvxiVGDU5HNjtJOZ0NfodC/WApNpbJ5JDsXN4DPlBYiG5pu/Yq0qAhsATDPbIjfVZLLfBvpmK68/z2w9Hu8FL1wo0IADt/23KoQ0qSEEGuOZptNP0N7nqeoMkc2lnb6VJBiXK97MJUkFEFTcY3HUDtek1XonWkfVfErOn2gCa9ODYdo/Jyp6pJJlarlYezQdZKXtJEbDgkAZUf+RNs/RJ0JabMjr6bHv97tMBWWV3mGqbZoZBE8BTBbu7D/Mp+mrcNPJDAq7HlgDgkEW6WIr15paipfdJBQfpxUl+bYIwrS9UNhx2x2rh0ZqDNx+uObJpPDeojfHzQ/ah9UaE3mF1QymtuVUVPEXrrLbN5tlb0dVvx4olMn56WBW8nqiUFCZ3rIz08W7xS5LRCptcOhscjDtq2F+NtuNrXGE1myZh7XPD1wdz/vecBiX5y0pg1+y1dFTlobaLluYLYMp3eFyvgkwi+/VQ3G1S58kmcNWXm4Vdn7bkzH1Jht78PGaaGiJVRJS+4ZhUpewqwHiknLRc8tEBWJ4p3ojiWLycDC8XL862Z6eQ99ibJBSYisNTXRKFYYH/XyT/A1BLAwQUAAAACABPcAxdzsQc1JMDAAATBwAAEgAAAHBhcGVyX2FsaWdubWVudC5tZJVUXW/cNhB8969YoK8Wzk6bwMU9GXaSGnBcw+cWBYriwJNWEmGKVMilP4r78Z0l765JAxjpw32IWu7OzszuD3RrZo5knB38xF6oi6aXo6MtyWj8uJ7xTVvCj6w3lvNaYvDDemMsPgFvPucXlnVnESvRsl8/jMbiXKJBnJSHo23TNF99kP/SiEksCD37idrRRNMKR5vEtmlJG+OMb7kj4zuy0+Hx0URrvCQykSnyHKLgdEu/JQZiprBJHB9xdHbWtMHlydPF1UVzeRlWb05Of6YUcmyZUjvyZOjJyhiyUB9ia/1QkJRLSXtWZtbW9xyj1gDmT9Y3k3nGdRCG+C3dhDjh/99MPRvJkRNJoD9Pjk//wtsPVih45cJ6GmLIc8Kze1kWehLKTujI2c6IDX4hnOSAKXJvRWqVCoWfQdEOR0r6BnczK9aP51c3+Llj42q1RsvUYzvNjlXbUoSApKbrDSr12S2/vDFxB35LkApi0SiETox2+T9AFhTwUJIiyKSUp7lUUIS/ZnGWo2K7SsHV0h9C1A4PvABZVvkqP5WYyFN45P3RPkvB9j/K3/MEa4CMEDuYqpB4OEPuvrctsgkcB63VY6vzPxBzC4DqH2rNrGoueut4EcMT6AAsryZc7pVETUwN8G8YSjKU8x0i28i13e1rCD9e3CCPmUc1EctTiA8EjaxYroDaME3ZQ4ByJXLlMI12VlJXwjP9SDkhutZtXIBY5EOHoz6GibyZOM1Gx2b1y3nz5u07WhX7Ly4hg/U189UtjSaNu6oBMnc2cqu6cDcw6Xo4TFXvwtMSNSj7w5mEObgwvJBN8A1IqhP5PersGNvS9er+k1LcctKOZB8A9A8sCy1LiT9nBv9ftN9xb7KTMnOn70CZT9xmsRBwN+iQLpWRooT11DGdHZPnR/TUxlCHyOxiG5V5MPOx5hE75JBTlfoYNqI0Oyuv93XHCVI2b0+qk4qHwcVoh7FxKOr2S4L4Wcp4FZdc2mQ2DoEZZR3wTGw8kGE2ScwmOxN11ak190OsUuHO7gEJzVwMqsT7IGsND11ud3vrwgEpObNhp+zVne+waxP9fn7z/p4wCWAarhceQrSF48Ms6F7F6jwI/tVCrUmXO1Ijq+n0xqSazLXQvu5r1F0353fXpS8wM3cBjIMZFq4sTRiM4d/eVeZo5UWJZJ8qj++fW5c7dam6//EU9thdX7RKgO330zTbeb/RvmHrH1BLAwQUAAAACABPcAxdAKH5Rz8AAAA+AAAAEAAAAHJlcXVpcmVtZW50cy50eHTLK80tqOQqSMxLSSzmKqhMLCrKL+cKqIx09PXhKk7OzM4s0c1JTSzK48rKT8rJTOIqyS9KzgAqLEktLuHiAgBQSwMEFAAAAAgAT3AMXSZNXZPRAgAA+AcAABcAAAB0cmFjZWFiaWxpdHlfbWF0cml4LmNzdq1Vy3ITMRC8+yt0phTb2AlxihMkBeVDSAqq4Lil7M6uh2glZUayMV/PSJsHSXzB+GTvQ9PdM9O9BHcJCXpwscJG09Olxj7Y8s9E9E5H4Fh5qmCNDbgaNMuDxKPZXH+FOhHjGtS1obsEUTXItV8DbZVxjeJ6Bb1RgXyLVg5SPWlMNOOwLVV5Umo/3HrEhabiCKGajWbH+ooaIGiUNTdgj2pvU+9UAxHqzE5tMK6U6W+wSxi3qjVoE+0FdaKXoi9iu1URe3nb9EEtL1gtr1XwFDnriF4YFG0WzK3pQA2MeB/Ed/rc98EQKFPHZOxDv4qmYAKQqq3B/lnt/Fvdd3T8k6UFlFwmrAwJeam0E+tUX6UoBSMZdKolU9rHajo+nRY90/FiOiCjHCQndNbGYjMsQWbAwWLkF/qebu5CXehrglZgO/IplMod5TnlHVEsZIQy1v+Fcaa/RXNjQbHJT2Wbix7og+CUo4MiHjbm2aTe74c5P9EXSV4Q4nAP5FIPhHWBRteiyypXcmXRdQUyEMjYamCWOy/QXj3bBfpOf/6w/KJaESRTHwZZXHaLIYhDNitwynnVYylzGNBTvWRvy4DUJ09ybkA+8s7KGFO0KJOU5PCyLYeBXOhLdJfmV1H6F1jWWlsM4pba550/kMbjhb7w0rgo6XFDw1A7MmFV7CASByMoLPEQt5PsNl17mXLHkzfjremtLnat5L3O5erjvnkNNR8dn4nlpUiXfOLH0GTZJGDVerGnbJOn197bkSyFYsUS3DmVd6/qfHQy1ecEWdIGXeM3rEov0bHIUewT1XBEfiPJ+shrME2x7OCWHVD/TuSt/pjQNvKJIHGi7Cu4Jnix5otmSx8ZOU9XkdlI+h6Qw0x/F5dKwv8G8veBMXnKpseMkBUjzyzjcCxTGcLgcDTm+oekYP529LKgzNKMhzNly4eG3IM/xPqAn0vMK+79LeyF/QdQSwMEFAAAAAgAT3AMXd/0TCMCBAAAFQgAABEAAABjb25maWdzL2Jhc2UueWFtbHVVXY/qNhB951dE9xlokoUs5K3aVW9X2lZXWto+VJVl7EniXsd2bQeW/vrOOAGyurdICDJfnpxzZuy8/RtErBdZZngPdfY6cLP6Hb+fn1avb4dfVp87G+KvEFdPL0/Pz/atzIv96lRgQgCQdbYp8W9vJeYeeYDFQvLIqd5X3rYaGD0GiHUmB9OadriAKXcFFtkXPwglpLQhlXTc/zNAxMQpg0nlWacMps7i2BTHnIcA/gSSjudGNRAiC8MRs+qbIaDzmnD1UXk0a34EzQQ3UqEFQp39+Uqm5ehZZk+ah7DMxPgTuW8hLrND+v0LC0TV4wm8dx+LHK7m5T2Cwp23JzDcCGDC6qE3FMww18XBE0yXZcZYsIPHiEYhckrOLN6e0UCFlAQTVaPAzwp9+s0Qf8hH/mmZ/aTtOXt5XmZvKTl7+bLMnrEVZXhU1qTnw7w5Yw0zQw9eCdYATx3di7+pXmnN/c+HwxeKDpilITCHLVCndVbmm11SBCJNb9h6OzjqGdM3+b76ABcSEoCNRaaYbY4fDIJ3h2oEOccB3cVu7rNetfgm+t7h7qN/iG6Id++++FBZdNBz1vHQYeGHbfHwsCuqbVOU5X5bVpvNsdpCteEgqu0OZLXZPj6U+fFRyk11bPa8hF1ZVMdK7B95uVgEp1UMpHc8FvGInivDGs8FIU3o5evHfJnl611O2J24Jqmg7xbEbMNmyUjhuiAwbrJiPEboXSQkyoQzBqIELuiYoNZwAl1n0Q+A/gn+gaiZKwoLIiyonnawQ0iaOmorvi4WOE0oUAEhKNPS22DYCXxkyjTKqHhh0bJeje7rMdJbx+SACAjq8gb43E19X77vwiNQECZ+41XB6gkii1OethMyaPhRk8KnqNQjCmoSNcGWF8luGGm959F6giwpK8s89DiBiLrtJ5as0ZdbOazT8/fxqOsIeG5aSBQig8U6EYjHauXqrOE6JKyJsrFBZ0UXaBjS45FH0bGg/qX52FbJRusMq0agbvdjHNeu46nN9WjQwL1BnG+B+fRenZI4+6wfdFQIOuAyw6ZQgxHcQz2fPw2mjaTvam5F2Sja0yQhj0aVptwEEENUCM191dy5GPfepCVchZGNgwyOKo8JYKSz6kZkfV87RPV973wb+HEp0ZheI2g+Ga204LjAo1ux0iH2q5auI4PX0fzuSNfRpM77Ic6iMJHgJLazMtKe02xw19HdAmKUzfg33SPjTpJpMNIMzXfcZrFohiSLpB46igRIQ0zYX9kf5TYnf1vQFWkdAofPSNqPkvd/LP6X6DOotsPrDwS/jNZkxsalAgIP9ceM9f1IP7aNK00iQVhaWMRgkjP7Tv2pVq/e8TVx6IUKaQMZJgbJJ9b/A1BLAwQUAAAACABPcAxdiAG9gdUAAACHAQAAGwAAAGNvbmZpZ3MvcGFwZXJfZmFpdGhmdWwueWFtbF2QUW7DMAxD/32KHGHAsJ9cxtBsOtGWyIakoO3tZwftivZPMCk90k3rD5LPYZr2mjFPjRo0FmJfy7GF0BRNa4IZy3La+ByjuZJjuc3TQiwhFJAfiohrF5JzleE2ukYIfW/I8+R6oL8pTOBfH0+h0GYvink/Zo80LAWqyLFB8mD3NRqEJ9Ww4R+aCqX6hg2LUluH+h5Heu+YUVj4PDCtZGuH9SSXqr/9jrPfxmJeXo2p7vshnM4s8dK/jCU67+iz5HoZ/R9F7snN0eJnPMPEnYQLzEP4A1BLAwQUAAAACABPcAxdcm6hI5kAAAAJAQAAHwAAAGNvbmZpZ3MvcHJhY3RpY2FsX2Jhc2VsaW5lLnlhbWxlj0EOwyAMBO+8gidUqnrhM8iBDaEigGynSn9fkko5tDfLO5q1O7cngjpj7doinO1MQXOg4icSlFxhTGd0bgEiuaYTzefoRZkU6e3sipipGjODdGN47HqKWj14od2j0lQQnZ2pCMaSIRX6uP0ml2PU41KEmUL7QxNTX474zy46JDL+QY3fU9H93UtYsJJP3LbuX1RypKPBmA9QSwMEFAAAAAgAT3AMXer/t2JFAAAARQAAAA8AAABzcmMvX19pbml0X18ucHlTUlJyd9b1CQ7x1XXPyC8u8UstUXD2dNZ1cckPNjIwtFQoSi0oyk8pTc5MykkFcopTE4uSMxQKMgtSczLzUvWUlJS4uABQSwMEFAAAAAgAT3AMXfh7knAiBQAAog4AAA0AAABzcmMvY29uZmlnLnB5lVdLb+M2EL77V7DsRQIU1Zui3YURF9hDCxTttnsoChSBITDSSOFWIlmSykZN8987HMqS7NjJxhdJnNc3b7q2umNFUfe+t1AUTHZGW8+EUtoLL7Vyq9V4Vmoz7N9vhbtt5c3+85PTalUHVUb4QNjr+YifkeAHI1WzP3+vhtVqVUHNKgBTdGAbSG6Egw2rZOmvnbdZYNplTN+BtbJ6QknZxQ9HR5sVw58F17eebQlwHvSHF9KeEkOtLfsbhozdibYHJtVkI5ceOpekUVH4yZpJJ5XzQpWQkEBGVlOMUbWkRbN5Az5B5enINWuaoV0jfYf4Fq4vCCOsdBKEFsPyjJpDNxfCFjCpauQew91qURWlVrVsKCJFSNiGYQjZf5StjHW6enqMj9+0AjQXHmdj7+2wiF3M9SC6ls7gvgTj2c90/KO1mAbhwumGsa+ZsaLpxIYpjR5hPtgFBsiAqkCVAwYaGcCB8kwr9otomha+MVZ/gtIzykHbToatkA6WdhL+cfjr/YdfgxoL//TSQsW8pmiwvZYYld5S2fOUUdkiOlIbYoXOB19yJ2oogmgSIjOHMc0tYHQ93PsEQesKC37Le19fvONpytDdh8fVWFRTkAMk7DUK6xy6fUGeMTlJf7HJhQ9HHTd3WKwaLCBZCQ/LKjmop3AwVtMxb3yc7NTZvykBW3bNx+DzjHFUJMLTmVZ6F94w40gvwWGXNXxH0p2kryCM9b/v5dDEk14MbzgKUcXjiGk3BT7Kb46K5c/QNrFWav5htBFFmUOAYRJu2MMo/cjHSWJFJFEbBubrPfzdNde9B1t4NKCKiZNPSBxWJ1RJjWn1yX1KrtwHxBNvyr5CN9f524yt83e7ZzCftsW63lFdezzGUsbjdkBVb9c0vFDnmh8kHYUmaXQpQnvq2AnuQtfFAgPfpXs3QxrW+ZpdnTRyxd7k6+cce9lW9PKGBnmyzt6ko0+N1b0prP4csiPV7AgVGrrhsGCwaaCYORe4F+JXW/YsxhOKJlBGO+nlHfB599QS2orQTjp5K26gLUpMCzUUlv9M87IDHHCdOUPHHrkDFZYQ9mHbd+qAio2tvESb9hRVYThVj9NAlkUNgm4Bh2yLFUa7ZVnrMZK088ipjF3v5sU1Jv94f7qMtdJ5mk5CDckpniysntgS05KOwscb9Wn3BlD5A+F5nNIgyCbTNYveXSjRQTCC3ezG3OAXhrYZFg4eTSDylI9DoNjz86lmJg2LoU6tNhHGofTAO6ikUGHKNaFfHp+rr2ODk1dRSQgkKZlwcOfBfMvn8TdrJ8pyXBHn7uDCg71Cx4sWaUE1/hbbAzv28oUURJ35keycCs9aEPh+ydOXzIYEVUBmD3rwi+xG4XOdOKUsyFBevcCd6Avbt8gTRi92pfNFbD8w/CXzvyucrlHfQtX2QEmojLK3FnsyMPfG0BY4i2mferwHGY0RKoxuZTmM+CqrTfFZqkp/fg26M0q3C3WvxdlYYW6LCvdvOV6fCB99o9grwB1p2u51vIzocLZyp3uLVTA5GRufbhqYCqniUjlBno7Cv5wiDApnRIk1cejF0/E2x4NgpOMUQ1iBMdY2UXZ5qE2THGk8M9Ci2qcTDWf3BXTGD+Mcw1DEa1nsbYL/3JUMv6J9h+Mf1+y/dCcL/+Xyqu+MG2UzuqkUeKVy2z8szWYwAieRtm6b8CyEbMPRWVAurA/hSim3P4n26N44/mfM3a24/O77ZDaa080Vkunemt/CfSUbzFKSrv4HUEsDBBQAAAAIAE9wDF2yZphnshIAACRIAAALAAAAc3JjL2RhdGEucHntHGtz28bxu37FFZ3OAAkISa6TcdkwU1e2M566iidO84XlYCDiSCECARgH2GJU/ffu7r0BkFLdtNNHNIkF3O3t7e3t+w7atPWOpemm7/qWpykrdk3ddiyrqrrLuqKuxMmJbmu3TdYKrt+vM3FdFlf69UdRV/p5l3XXJxtEva7Lkq8JkcZ9UfdVx1vZn2ddti4zIbjpN00SogFcMI3ufWtQd/umqLa6/Xm1j9lrwJtdlVw9dXUbs3f8fc+rNTfrqPpds2eZYFWjm5qsyqEB/mvyk5Ou3c9PGPzo3n3WtvXHBFYPqDoCe3/Cb9e86dhrgnkJAO2csV+zps22u2zOqhrW/oG3bMb4LW/XheA5u9qzP2XbbclPC2DBtiUOM159KNq62vGqo2mb92zBLusKSKaFJuu62hRmpfItRfbHrKyzPJUtJycnb57/8eWb9OL55YvXL55///Id4AmDN9kVL4OYBaV+uEDu4sNaP3SwubzDp+/lU3Ty9rtvf3h5+fzy4mV68e2bv/z5UmJL03XWkLDk2R4HpKmo+3bN001R8rTIvTZgGzZFQFvON2ydVXVVrLMSSC77XZVW2Y6H+M+cia6N2Oxr/C2533KYpsJ3gogSeCqacISr+IkrdCJUv+dm15cwaEV4y0J09Caxm+GwquU0XfI5Ypu6ZfKZFZV6EiuJBWVZAAol1KHBFKn+EuYl6V8wAbvHc1oM4cSHWKKQiBFXUnR8J8KIFRvV9TU7l8ioReOTqyA+ZSBb7Ies7DmJYbgJLojGGc3kkJBtgET28RqmEE225iCk7Q4ZSHI4Z3cW9j6I3E0wy5pi/qZFbtG/c1Cg5AUo8Ct8I767DWPWT+4ibpXEl6imSFMj+rKDYbqz2YdujwZ3EbvLkFBqDTnvwDKlfVWAqKi5DwhQjOjyAiwTH3c1fdvUYkqC3YUK3oXHRFatcANbniuRlBNK8TNvReXQQiIx6JIzrLTElLwKCWnEfrVg50fF5uVtAxwBO8Vvs3VX7hnYIHan1nevdYBs0h3tkaUkuo8V7Xf0ayA/1LY8WynWiw6tdCqyXUM2QxyTnne8LbiS910hBBp9ZJCixlPO8HHmSKkSDgI/hwNpfsM0Nc1xHSMfydYtR96bpfze0Ni0YP+rDASFGam6U52aPU22RwuOltXMRaQsxwtZJZkAn8fDAA1htQ2iBLrKKguDrxTarxVa/PmcBfPAeRuiVbywWF9X3ZdPAemjZ/F2WK0k2WVNWGa7qzxjH5Bdcx0nJOI6e/LFlyG1JqA/dQ6T9N1m9iyIouSa3+bFloNQRdFAStbXfJeRvzuonjmSPGn2jTJaVmOkkuQQBShL8xN4FYVYY4qiGNyyQDeXiXVRLF5lpQBjLTgEARhXiEUYxChb8yDy+DBYrWbLsfXCcv9ggh7QhPonXi2+b3senVATQ5UA+4Ghj9KDtq67uYyF8BVHp4O2XVYVG8A/bFdRDIkVcKzrQW6X2BuzJEm0hqZGtdNczk54RCh41q6vHaQxu4Zgxlo/8rMSK7THBLOyTtdY0Wk41OyVTz95sQULNB2i3+2ydp/gNgbKaLZMtaIuOyQm7basr0IPV2TVGlRdYwO2QPyV0GRgKAM9RAQWXHr0qiuqnlvTANOgfffxyF8GyGwQQNKvU7keq6BiXbe4zDPTUtYfwVsvKADCMVFCLWHkko+81+24dnr0KZaYPwfjf3bmDjU0JfwW9gJijgPjvvCGhZp+yyF8A7XGnYXYIW05xqlyf6KHkD+xyK10JFnT8CoPQwKLiWW+mqlYyg6J2Q3fL5TlwRhqzkL8BU4nJidIL+cr3JmO9LvlEJ4LrjRNxQSFoKhdC31ITMqLluSb/U1JvBZF6EiN9IPFUP5T6xb0wP6dPyO9GCuxI6cAhh1mOhPwoYtx5XnMTemaXsGMl3X3Cv2s9FDOKIPNbRxv4aSCTW2gFnmLjLo4mMj5xG4C6GPMyZitnqSTs7X2w5OmQzzwgPBnE1zWuFf9GlOYWQMBIW8/oLdWc7OPRXdtbI84neIJgzSq/sjuHNrvA2+qaMgqSzjII0ihb7UnjYJnvx0Iu2djcw5gZFFDpR5WxaUdDD7TSSy4ICMVqB0eGgoVgf+hL9AjofsO0pNixyd47cSSdz6SexB0mUbLOIRJwsF03RlqbSw5Js3htGcQXPUKrTgpyfKYGftM07oPgVye4v6GWHKQzo00Ny/WlDnGWGRYeckphRLo4AUNSghHx2+BdejwQbAWxuWraZShtIpgNpRQgNlwl6JWITP8+YCUg9RpZ7gYLEokvlQdVHrCYk36YxAdcgCSIIrhHoNGJeBygIui2DgEJVuwzRChZl0vAhLXoMEiUh4wiAWGcBwFNKVsWgKfHZXkTaB2AJ09+gjkgMMM0GSyRjgjhPS2R0f1QPv6BpXxzkwTuMG8pAQdFOiYYvxyAmIFXgpBHBFYBkadHHgBoFFsJ6vbYltUtpxhJvSUlGYnPi8PjDhOwGAMEmF106Wm75q++wdomYB/gBJ3xEE6nEQC58e4Sk/pduFU2HeA6x6kQn8vw9CsKMGr0MbLmhblOsIp9qgGzNRJRtxqj+wD/4ASql7OV/da+DX2R4quViwKWFtI5zEv3WXdGizbncalBXa4k4cKMwekxXAcF9pCCla0YPhhkeMaohd7G1CVhGv0NnPWE6QyIAT9Xk15/gEPvtN4Rym4Sc6piKF5BBzRpGiOUK0U2DBZJBou3zPSS1lntREPcMcUXyVypBVs1675xAnM+OEkpsOvvrhmSFobkH/15OiHNWTB3LF3Iw3CwfTg9NgK6kg25iPxcsYpXlE7QNKr0+2sVIOYJq16KmGF0L1ri6selRRrG9u27pu0AP6suQi7ustKisiBl1zgVtObTVnhRckWSCaBs698TyGZuXQqv7B/u6KSyDHIuA3PDfrJfAVTCjRjLYY34W0ki8u3KPxVk5RFRXXZ8CxWFMzYuaoQR7YsIktNbgzjRCyx6hdqsYK7Kx0XY2GxzXtWCDpwGNoW53AjDNQxCMIaxe1q8uisu+bqZMOEdyqwMIUuSy1wrXmfKLhXmnwJp5gnd08o/rqDk6rf2f0Vhut45JSseVGGav0QTDz54ksd5nrSgAnJcXE5OqNPo1JpLFUh4qU3lMIdMxa2Pue3cs/pEffdm9hUINEiSpwjAXT3UBUDqLaHbM3xrGiddeGSBiddncqzrVDOSq04q8QNNqPYVpBip0SOSoUVCVQ5loXYr41Q+VVKU4mX3WG1UHCQXcO09S7FEI0vUAijBHVAThRGCWZe+i1v68aZe3jmMDhj0HpAR19yvtzk60rYxnG0DEMldWkDGT7uj9QRlYwrPYlPSFNkZcrldTwItZW1IJp0PcuFd2pZLV/Xba6BBmgsmAkyTJnTHFqxv5GC6oNB7WyJfzEtGHeVg6jiuSdX0bWf54w375ApkbHWkFuRtCbsczmtDbJs1DBxcuM6/OECx3Znig0YjcgnA8ZLU8EXGC6NOPdwkPCOVqczk7zYbCDSh0gA13/v1NFV3dwU+IMV1eQmDzAGo0wda9CpxEH33nnEBsjqgCpKOqcswQ9/4GlXq22V5TDoaEr0FsFf/4qV6NPACXYJk1YOMEDohq1CT8FhrX8O+eCUXRQpGHhIymD3NTqL4l4fde6uiornnhmSbDhmZfQwj8NJDtoHqo8uM0qyau9Wn0wfCgbK6SNQQLATRs4GTAqENI4Ghz05PdW4KPujgA2NxZ2lRAuMibq93G9X57ijej8CN6Ia6FgwP6B6zhDYBQ2GdtUNlzBoSCe2XbPIBZaJ41xLo5vJaAusRtn6iR+CUC1829e98E2IMcIqJHFt77TJJd4VVbFDV9tXRLrbld3KLiOQurj6VJnrcWATBJAHQGiC+9UibtEVa2ZJZqKkWAALfpA/Aj5IFTrMDSAlg7AHoq8s/xH0q1rvE0D274yWYBpHCiIKRBmZ+y4csikahqnDo0otYww88ojJbNeDtF5x1tSiQCvzc8RrhpZjYZQB0gEcDKT1jfY6in1unJ5OM0Jn4/+eiO8K9it934PG4VHCLkPbhsc1efEB9D10KTZBKqqiN60OTx8TRKC/l7sEOZldnowiref38Vu5IFIBm6WbHHkXapzsK7sOz2tLYF/KpMUfHIVJA7pgx0Jgh+7IGXjDlewgiwgPcI0m9kghwEdQooUI/HRLWRrw/0zyXyKHzApxWeTyMBZABwe4m+AO7dT9nAIDOh+EZ2cV98H4cNec7Fql/GhowRM7LD+kV3vwHKGEXc6fYQJ/VWyDiP2Ghf4CPtdXgPAHAuJthZUohS+0yA9qBvtsusMgNYyCLfAmiH1eHopwJoJ1mYGQlQ0VKmK5m5TYQwhMeJQWfFLC80CkYWwwqNOj7oioQ+gH7pvYYtcnXSgZRCAXjnOS0Sl4j3/kOom+bSNzMky3ZB0x1HTG7AY4swhk/BocTMOcDMtx8kLdrvhZsy3J5n/C9f8/ZWpHIy7905iYy1jO4yGlAcFwUqd34+6R9ZgAmXDethb/350wjrXzl9zx8bmjF4Z7RoXC8F/yyU/IJ6dM88+TYw5VXQ05YgGCseoPJzxqHP4T0tqrvihlKTEFb2sNqzmk1mdpw4N419/5l1jjwy7y4Om9eyZCsetg9qV/aiI90vCkZHrg6DxFDiZnBNmx9jOJ4PjVRiovQ0Iusy77nC+WVYOp0RVv1XlfUW1AIrp9au7BWymVE8gNhXEg+NUmVBOpYGqFoSB9jhHSTAuA20DQ0X35NGZVJoMXbISwJxopnR+JKcyJa/jvDxF5w/fqcJawwKs6mkVMA/jREe29GxkOkeJqCSqawKyYW4gq0zbkAHb60CZP3YNCkCrgFp71YavS3uVwQyGJoNsOYgHWgUM8G+DZ4X5TtKJzw+FC3Owdd7kcbNvwWrVv8/9SYRoEFusMfccrvAz1+gU+vpNXel6/dV7eQhiKry+AzqKSNg8BfJRurx7xtq27GoiA5+EyY/bZ+GzZBhpOOG7YTr0r1w44FtW9lOb4T+UuXSs1cY1DXlIaR3GecVSqKE2Za1LHSqrPaFfLwAFfRSOT7ppC2eLCDO5bWCATaY2ACRlGZIcBpUkI5McipOB0VwJb3S9l1EVqR+rlQC3v9w+cAet9HkOhuilfgFR4YoSkgMpFj9LDpTvRKqFehZ9SoSqTt76NjtoI5ZEn1ONF2C7S8FRfxLALoorMQPs9i+G7adcIWSR+uzNgYNvcab0Ol+88u8m2PK1biGGwQtoZw4Fe1jUkzqgmQ+e/LjNYQ55ueEafjGlZfPbUAQWZ78f3BgzsISWRVxFWR64uOeoiL+3IQJ3CJMjSKYT69govgEKUqWJ5yC+yvrsGnB0Fzr9nRYeNVJ6oQfWpivvsqc4lksC/hvARBnL3FqF7bVh9DDBHX0/e36Yy8gaTvgZs66uyXd8q393kBVomfBFkzvHSMShsWt841l0VsOkCoEKAVe5U9JtNcRuqJvmG34kknbkxYoYmciV0l9H5YEItIaYcseoWT6a/kwBWZH3ZLfCjAIQY3IUcTKWTEUmYrj5kfV500we8k7eyHX5N901fonx0FeFAVcG5mUN5p7yiQ/d4Vzb4o8Lw4G65zUDUcmK/xcXlXosOVgcAD1wNdKvi5jbY4sGLqN6lo8iJcPUVk9TJTg4fxhsWHCB6lJ04q9MwEI3/CCtDbac0wVuSCtWxtD2O3/V6Y0M70eIql3cB38qPBHH02YVWF2f1NPKeaqxJeWjogH169KD5MQTgtuklajROWON8KgzGznkL1a66jl1fBPZuSU4ZXO8KpBdD6NuyqXeVzOJwmtXAe/+WlN4jyccRSzzL8GBh8r/KSoiON791uuj9/86CPLin/0JrYnYBm9XcEDRvu+spCFuMGFU7frFO/yvWiQbqCmeqvhYC6XLF1JwaYThMIdmnGzaZFGTtFisCYG70n9xILrHsiVdD9bes0IiHzQbgebvt8Q9IvKWeMOdi3RYNrmURPEdzSZcOJr98unh9MXvxon735Oz8d4dvbwJ0kuU5EkcThcFshkAzEMbABnzB6Y3+AxcgDceHS4k5hOBj3d4Aeadv+qya/QD/f3Mxe/Pu+z/PvrmuRXfJuxkQrumefTg/lejEKXmH4zNL+Qpic37nBM8HhmDNVY87Cqgs0AzMwYzMQcyotmW+Zzs0Dk3RI2Hln+uYqYxz8B02jXAlSQkXHu2Hg7QDAeSBvYW2ptKxyvAWYr/6WyQxjUyQKZ55LzayY2iGdQrln934bnDKeEsRH+PzJ8NLiUcnGFl8gxfePFySs7qG+iiqx9f9saA2xiZ9eeoYgRTJmMxyaLRx6pI0a/xj5vK8aTFFdnM0jf9YjkbXyWHVKX0TnQIpCzwERyFJU/XNs5SYk78DUEsDBBQAAAAIAE9wDF1iW+oAiA0AAAwyAAAWAAAAc3JjL2dyYXBoX3NlcXVlbmNlcy5webVa62/jxhH/7r9iyw8FiaNV+9IGhRIGPTQPFEgPQS/pF1Ug1uJKZi2RDB9n61z9752ZfXNX8rkPIzmR3J3Hzs7O/GbIbd8eWFlup3HqRVmy+tC1/ch407QjH+u2Ga6u1LN7Ptzv6zt9+8+hba62SF7xkW/2fBjEoOnNIzmj4yOS6tGf4FYOjMeubnb6+bvmaKQ106E7Mj6wptOPOt5U8AD+66orSb9AQZr8sa9HUVq9Fl0vur7diGFwhPxkHorqQ7evR1jh1YeffvzLz+X7d3/97gMrWJqMPa+bJGfJR76vKzIE3o1iGJMM5v/JLDAFUZ9EU/zcTyK7okfs3b7eNYr78orBX98+DktQe/Et0H3f84Ogx09LWN4CVtX3/EhPjt6TlwR9EL9OotmIH3re3f8smqHtBylwINlsGHt5qyaWoUQzdAyGRt7vxBgZGPih24uyroZwqJ16YAYLjg7v+nbqoiNNW4kSfUycGXusmwq4dmMfjItqB9o0lQiXR0MvkY71QQQjBzFytP6SVfVmXIEpc/TQNexKJbaM4yaXneNNJRk9JeIt7rG/4zkNmNnLiCfms73LrzJ2/U3En+qtnAWWGVndMMd/5QRyOl4Pgv2d7yfxXd+3fbpNfmkemvax0SKe6fcELq2dFLyfVF/s282KrlYJTUrWrCgk3Xqxabtjmi16MYB7kN3Tqm875Zvk2cAI7MkHsmcKbsTHsU/NenO2TZT08inJJNHxFURHTWRMAeqpc2tNcODDg8/UMFvQXNB+X4u+xIk5qyAeieKubfeZYQH896JJcULGflPQDVoqs1Kixk4C/qweaL+43E72WI/3jGZheEKeiRWrNgN/aC+QwfqSxZWepJrW8yljbR8+PmYXnMRb1TZxnZSRCSEC98JbyBaEqH1ZksbFs5F5ytmTvH3C66O8PmanxAhS3icgBTWerxOHAv9BJk9IfczU8StFU3Vt3YwUM9KPuAB5ZlgDXjt0fCPv6QTB79KVonLZYrjnb//wJRyMZ0N0+sfNM3E7JQuIixB50mQat9d/BIdb3Iunqt5BFkiNHpQhjDZaEQgUJBh9yRxZiAb10HA5x90DqdT3fD/IvDCKJ/Bn1FrNXcBl3aWepZB1ijMzSNcV0YCnPIo+zXRYeE4aTomraRuBv183/JvkpFXftM1Y76Z2Gsp+aobUxm4vBOcg8depBnwABIPYTGP9EZZIxwTXuK+HcTVOkBFWYIEcBI/r9dJ1Sodxhuf0Jlj7aq3no+oxeQFFepMHzDPJ5q4X/GGQB3+75yMs/5Po2xRuq3q79fWBQ3GbsTfsVpK2Ey67FoocVNjwUTTwf5qubta5Yp6z1Vz4OvP2Z5WiPwyQRMeMjALXbZdldFzocc7wCW7UJ9hcK3m1vL5FQfbB7XKd6cRzN9V7yDU6be8w95ejTP7y9KpzufQOU5hb8B6Wt6138xSn8s55ePF/yT/DKLovwOpSJ8g7eJ/IHQVb7yBaFsqOMADj2gRyMFlrNn1diTNT5aCZKndv0+6nQ0MHTs6Wj02EkeNKkwpOf90QJAwI3bE4tQkzlsgLZKWZoAjUUahg/rOxrErJcKA1nsJrg8rwpizVKrZ1+EyisyQ3DD1D5JE1yqknCYxqCahhBQCoRZUaHa8BTY6pcr8F5S5JPthkragvusmfqf5gGzhpo2AfwE7sC6Zc/CsjX7EGN1JPtCOhd870MCgGgAAm+RQzA519nbgu6aPOkUzKGDUhZ1ONw9ppHNDbiElh/TlA3RQkbUwFNKVjngvAz8+yWJzmYIx1WDh4/IKgGTQ/P9NB6TQJjOWMekj93LiF3Dhws7ZgW6oxRBdCExzvuzDrvACJ5iN0iJg6wOnKXewRlbww4t9IS92DY8/TI43ScNtXQh5Jz9WhIoO7tCzJnGXXDjWuoUD02fNmJ1LMGC5JpiEnTAE9vvy9OiZWgLpa4EkrCQsMFqGtoqfcDQnzEw9Z5QHWWkBs5Xd7ITGYlIlZSZPm8goju1aAHtwdU5c9KlUQanH…2017 tokens truncated…WRXNQE+giA0LKCdzzrDx4DdGNLNgOEAKtaCWEpD9tQ3Q5IQhMvNSVlQleCNhtSKIk0bJHWQQJ4mQPSzqCRpBrAuIOPtCUJq76oARo1bYkTgXioHbIorrxbiweWN9qta0MxQAe05Fg3uDMQRu+k3igo722k4Py3F9qoF13vMs/N6iyxVj7IWI67ifdkefJoIKSldOPeLOzrJddNmWXQnfsamLKYWXdngQ7hWVYpmB+NTEjZXCSUbWEW5Hxe14ktHRZXSM8DhKHuGr8DZL4Tz7oNodFvoh2hni5JmqThDsaNg9Zc1G+fPo38W/urE+eT8mcotFFRhOFLpcWrhXaswv91QIUx0gENJv/UhRwdGjhJkVPITI2ycYzQYyYm9J9O3XsrlU9k/BxXv5j4bkNWQQnkKSLxInIPVjLgEdaQZyhPIEA66o6LS+FgBbP8KUeOCMzamzJ9T4T6gIoXg0sGCfBH/zJSrnLF1jA2YDOyZXeUM6T/vbLx1h46fSkCcP/UhGzDhi0wb150KSqR3SJVb+MjJsotPXKXMEVvOWB+fdVLwD5XccpSe8YkydpK9ljgMsioW0loO+znBOOv8uxidmK949VrTm5R1YdYUft07kvri9pi06CiUahY4jCya5hQn/6QpBTN08DPH1j4iXCMIpheBiMAKsLr+H4Go4uE7ubghsWpfVgVZPQw9Jdj5MoPv/A1BLAwQUAAAACABPcAxdtpQP8FgKAABUIgAADQAAAHNyYy9zcGxpdHMucHm9WluP3LYVfp9fQeihkGytvOs4aTrxBDXapAjgBEGc9mUxEDgStauuhlJEyt7Ndv97z+FFPLrM2G6AGgGyQ54bz/UjZ6q+PbI8rwY99CLPWX3s2l4zLmWrua5bqTYbt3bL1W1TH/zHf6tWbipkL7nmRcOVEsrzj0uWouMaWf3uz/DRbuiHrpY3fv2NfEjZD1r0/NCIUa8cjt0D44rJzi91XJawAP915Wazeffz2x9+zX968+N379iOxZHueS2jlEXveVOX5hj4SQulowToS1GxI78TedFKXd8M7aDym74durwuVVz1/Ci2IDn7O5zie/yUMrvdtx/UltVSJ+ziW6R4J/paqO2Gwb9e/DbUvSjBhOsoz1U79IXIq7oRIBb1j2sgBpf2hu1YK4U+AK6ibYajZFXbM/dnLYPYuvKrEBvcMYZaIbDn5FhbjD28VoL9izeD+K7v2z6uor+ZuLKiF1wLFk5vj6e+GY15dH88gb+cfDh1HLyQsNc7dnlGWRRo2XFQmh0E61pV6/q9cEKDNxScHryp2xyCDS4tbBCuly5LmUDxahcZjVGScQVJJOII7PvqVTA3ptJfs0sglA9xcs7imbLRbNnKCyluODH90LTFHVpNtbx4MXeSywuoLRetlcTYjydQugeXRwl7zqLt1qjYRfDBKluQuUTOlRClKF0Ct30p+nhM5i1raqWvgQU8h4Qhe8eNLbVSQXmJMh69NEpKx6U78bBr+PFQcru79Z0hU7f85ZdfQZ49oqqn7aPZf4oyIYu2BNMHXV18HSVJdivuy/oG6jFOrODxNBrr0qZjDvmAodC8vxHa2kTrcKy/dDMxlR7a7FgBjqtqWq7t+uiQdGNcogT1iHGlKeezLrZ+taEublsl5HYUhAkClrvNoe+F1LB2aT5jmRspWMtOWUhPrHYjDXpxyfhBxZ7/gp7H1CHdfT7LwWvz5z6ZsQVFxjTsBHIQ46JVnfGytKKSsOP17E4o8gVoGk04weKARIXbu74M3I2QsaVI2G5nPjqqxIibLHzLrhZye3Fs34tR9MXVflKNlsonXVljUR0GnBS5Ktpe2GxbGwVmo+EH0eS2HUO4de8SqmtqvVx2jgdphRmpW1bWhcmQ1Kbj3iWgHrpGXNsMJTQwF/cuKTWM5ca3zCO/j69S4wtjaWKPCOfuedMAge051Nhlt8neYwuE7UFqFcu2P8LQ/F3sfu0H4fo0OgTTNrOJWwrN62ZyCrQQKB4j4wEVbdnj09OY5WYRs5wM6hAwNRygSLy1GTS7a2s39eYes8As7EdG8OYAzvBuZTZNrLiEvSC+Glk8bW6mCLOlM5dzsQiYNWUfqsC65PluLvAZe+WcZBxF0gpbgbHsD4cj5J8/hEmYOPZxv5gohkqoZSnu/XZmPkHi1U2TG2U7CCz0ZHRFkh0Fl3ESNDkgkBuNduKZCYF9bSIROwwuTnTbnRW/0QM8h8x6+SX4DsM31Rc4XdJd+wzbu5hg1k16WYQBhwQkuZBOCXzMgGgW+hnhLA2A/kRinFBgzwds04UZtXXFpAVhEHKIB+SIFqMU4rOZiKnTgHS6EKifJrMeg5F6z0Iv/OsI26GftL8L6bLOLLF3eNRfhBoavT3VHZ14BOnz9uB6LUiqb6QbpTaW/1uvbcE1fW6g/hiPyXQP4H/cz9sqJ3wT8hnCXyAE/FjA6EGZIudai2OnLS3k4NXLr10PX7gJpxk5wgS9n0Xrb5GLYv4KmgKY80jFeYRujjX2gjXnJHOv1FISlo+7azLaoWcArLZaX7Mr+ASVHDYWagzNOfRt/DZWSrg0gKviy/Qqifz8Rs+O06Jou4eY7lxHHplFezMiT13zLDm92SUkDxT0XGC3RJlZOzzEQXZq+uDue94okWRIHRN27JOIQeMgzLZC3+4hi5MJ1rFsgObYF+ec9EYDNchg+rYXwivj0FP9NdG5ad6oJm3SXZC3LnzPWIzxu1gEjTROepUOfHMGQm8u21tmBRt6u2k70AF285WKZ/9hP7USPY//C6QegthUhbte5S9iuHsKkCyFIRpxpWvu1lzeiBhx7LKyE4KR3Zq5BzALfA3oh9HlpT1jV5ev/vzyLyMPOiAfs+Fj9xr/LySMy0xl8Z1NxATULEaQdTXc76iVo9AwQW0ljxZZvD69hbhNyEm75FoVOcp+Jg4tRWkk0Ue8QfQFK0jKfLZziMp0ZcP5a7LzMd+RtJ5N8UnMn7OrdMWldpiZnHCdgvQjSuBgi2lJrvoWNAb5BobQxrIaZnm8cFySMiqVnORzRZP4zoSa1JqitxEwYOBWLk5eTTqZeqPYdBGE4C3IO1vor0nVT2+qoXGkhCad9AGE9aMRM5DjvCJ6TUSxWplUN90Cb5cTaWRzNkFBkfsgLJSBLqjFTV/rhzhIp2PFuwTBvX3zKvpWKc0PhIEGKCVmfuTmkAS0pPIPtb7NK/EBZ/ctgEkzL0LFzZ95QEZsOBPTEMyfY0OYWJ1Z9co/CcZLEmvnHq7llzAahyNcJ3CubULtWIQ4nUlrkMWPmtU5dAqmANOZuSSqShT4khcScAXXEwaHD4ZGIJBfvN9Ba4ChBLN59nr44nH2FviURAupxl+RwZAn4p/JQda/DYAu6DBeTisnZGWMETYlGji7KG3RAAupoEBFa9rTuFogVGOmT9xNCFwidtAYhcTKOJWO/mZzPmmJ5K6vW9ScQ1niKaJ/uPnlLMKK5YO+RSLzaPuNS2g4F3yu6sJWL5Dh4S4gJSAbs4gCFHdFImA+pq3HJrB/tDzZBFYQjrkjTHHKytcHBGeOfTNS/Ni5N+P/+zcHPzpdxpiL4GrXDbaLbwwGeSfbD3L6YODS2w0Y/2JAXoUCKHb8Z436p9Nh35ckyEZDHKc3xCEToacY2PCYl9p4/u4UBqB7ckoZrUgK4U8+b5FUcmH7oxaQ6H+2Cc56iFqvxMkLQT7B+OOtxFh+7VDLnv2J0VWKn2irsQLdJeDTRFkYu37nOCWJqj8nbhKJz/cDCSG1frL8yZ74FGGf6ouTFpwV+OSLDL8SW8kN+wapcG7jfQmI1jwXqOZF+gtMtfroynQC36roreB3/EYgIDOzaOsuHbvHFUOeUncK2F4z4Slg08nrPonoitRou1YQdE6uqAKmteUVrjK3Ixs4zKONq2hSvuUA5QpDCLqi+1qSioGZNaC6qEOgWo5jyc6aD9B2/aDhPUwzQCz+MWNL51WK0KkbYHbXvXk2g+swfvFuBhCC2a1/NwIiqALciwNLQnaz4x2sxB3Hb36UeRRMmbgHnJC3d+Rl2k7FHL/tB4FO8guAAjbmud3P8JcDNnAIUdqe9w/mAjUyZwYIqKGq6vs4MvSZPnb+acMzZdYXWtzr2NCUw7HzvsisvJThZVTq3UuwWCr8kQNXRV27lxtcLNoS5tbOfzk50wFiGl6ImJhnSY5c1hVCAg+IYQyPUVydx9NSIBlB5wsd+Hle8M78MKPkD5NfEJz5VUGoCDL3DRowbqnGZ9o9eTWzy9fzM+3xK/lCvY9DLG3mecIMNiPr43vn0s3mv1BLAwQUAAAACABPcAxdVBAmQDUFAADOEAAAEgAAAHNyYy9zdGVwMl9zbW9rZS5weZVX3WvkNhB/379C+Mlb1t4k9KE9cKHk6FFIQ2iOvixBKLa8q4stu5KcXAj5329Gkm3Z+5G9hUA8mvnN92hUqqYmlJad6RSnlIi6bZQhTMrGMCMaqReLnqa2LVOa99/fdCMXJcq3zOwq8dgL38Hnwp2keSNLse1P3BfdMb1bkaphBXUUz1wwwwYLukIYqlndVrygeKK5WZEXJQyno+q0VbxVTc61FnLQc8PZE9vye1byu+G8UV5Et5UwelAEkltJt6rpWuqOejX2izJlRMlyA5FYFLwkNghA3ep4SZI/hrikt6zmumU5/7Qg8LNERbKR4U+17WouzZ09iQuucyVajHIW/dtJYnac3BvekiviHSc63/GarSvn0Hrqra6bJ04M1yZaBipTVhRon9UVR0mC0UsKoaIVAQdYV5ksWgPetuJrIdvulLg9wB/gNJ0BZoc00PcQXxr1BNatbzomk//g78t1cnP/9Z/ky67R5pab5Prv68+fm/uri8vfk+fLtYPVaw2uX1HrlMc/6ZUrndAnR9HrR6iV9JXV1emw1E3BT6C0CpIuclZRxKuEPAfT5U0nLVdJKSpwhJjXlmdCmlHF1cWvv51G4f93XOY8sVWZqOZFB0AfiPLiXF4DH5COvKm6Wnq/FIdJIHuJsNZ9+ddMSFf4t430pY4MUOgTbqT77s/CXo/x3M+FlZVMMRH+NJTbRFi40cMm8lGlEFVqo/oAmOCdw5qfOgxROvA+lr7DMZZEaALzLXDgoNJ9wZnefYa5aujgk7qgmb/x3Dh1kLgZPi+mgC5h1CXsPC8q9sgrmjNZCCBx58JmH+3BIrhWBA6c4c4KR6LQ9MuAJa2fgBJDxqGWdPZVdXxF+HehDW2e7KfjdrlZEXAUU7PyBFozKUoYXDgeD016pxs/UPOqN2xNIssOFR7Wi3fShSXrlQ3Ou/p2HuqurpkSHOt140hloxAfZrWQQ/zcTYARtEfUKKh7WtqZALdi9DCG3F0TEqY/YJZOkr5hGqEuZBGXUP0mtjBL8gu5vLhYLt+jmfgQ+cHTEXbGqriGKYKR27+6xnkdRH9CC2OShR9TtkNeZ6EnU/ZnVmF9AdPATZuSBiheeCIVFuwY8I+xfC7738yWsR1xAsZn9fUMAltvIrzXqDOBocEoM4bXrZnqHp3bZ5xgLYf/Dq4gcVgBq0npjJJtsPJAlRxZhuJ+BJ92M4S1kjDSsomKtAQbIC1SQyPVfR2GhqaQxhp6P6y2w+bCLH/mcejWatSb1twwzOIo67r5FSx6m6TjYNNGn8jxAo7wEgKOA5GwJw8zdgetd6xFqQoGXzwa6g6/p/Z4rigo7yPiAccRDNz5jirHsyNyJWd20c9hMBkQ3Y/tZsZz2G9sGkhu3TzDyGYac1UJrjRA7nf4ISUBzCOHuuE9hENl1azF8Zd8hMRKTPppoHlE/GZNNTx3OrR/UrhQmvBG2ETQI3wLDflqO8OxziNjq1vTWtjlnOJbw0+rw3E+zn8O8lgjPwEfCJ1lPb4sfsJ4ZA9w32ezDF9tk97G69yu+9T3cYoscLf7z3mfw62dsrblcKdOOAL4EVh10u+UPeo4IvyOP7T7an5in6jDsf3yMzKon2i2y2DxTCmO9/2ElQfdDwwdyjJqGaYgeHhFflVytQ89ENgxbEA9jRp4z1f+stt3wdIDsZAcskNQbY/06Zg42Cq8StCHtOjqVsdvB8zfx3jHK6iANTK7ghVSapw9TOdCZH+xSvMlPjxgAaZ2FaKUZBmJKMVnCKWR28Lcm2Sx+AFQSwMEFAAAAAgAT3AMXRz+juZGBwAAuBcAABIAAABzcmMvc3RlcDNfc21va2UucHmtWFlv2zgQfvev4OpJXthyejzsBvACRXqgQBoEm6AvQUDQFmWzkUQtSTXNBvnvO8OhTh9JFzUQICKHc/Kbg5nRBeM8q11tJOdMFZU2jomy1E44pUs7mTRrZlMJY2Xz/c3qcpLh+Uq4ba5WzeFL+JzQTrLWZaY2zQ598a2w2xnLtUg5rQTiVDjRalCnyuG2U5ta15Zb+U8ty7XkSGWlm7F7o5zknRrJxohq2xLahlU8YfC7ujz/fM0v3n35cDXzCyJXm5JXRlZGA7WVKbdVrhztrmqVp51QYu1kabWxRGHFd3mUgD68DUasHZntd76DbDCjdzyX4k5s5GwyDbZ0eqmy9d85UV2JTF62+9qEI1771moBJ8G+jdF1RYbZxmX+iwvjVAZ6QYQnqcyYDy6sbmw8ZfO/2ngnF6KQthJreeqV94uGLTuCd2ZTF7J0l34nTqVdG1Xh7VlGf9clc1vJrpys2Bvm3bRozAYfFlUuU2YLfSfBYdZF056QRKQpauS5x9F8jrGfp8pEMwYqizp3y2gBLtnkcqHKqj523G/gD/jo2gExcWrXdzjea3MH3l+c16Kcf4W/T2fz86vrL/NPW23dhXTzs89n79/rq9cnr/6cf3+1ILZ2YcHYN9wbFfgftYpA0LeJVuxiBTc9eRBFftwthU7lES4VXj+1FjlHfrkqX8KTQmPnlTTzTOVgCHMPlVyq0nUiXp+8/eM4lxDpub+Hc6PvbY/Ry47msty47U8fs86oVL78mExfSuvgA4K/1nldlMe9mNV5Pg8ZC9hjHBAV1mlIt87U8pkgOCNF4f1vjx03EvJ32XDpIzmAuxCqJFhf6DIAGQkAxgNqXFeZ30pQ9ybbMm1okTTiXqPTFjpGKCvZV5HX8oMx2nRg84AL4F/XxoBh+UObAG2bAbpMD7o6aRRQ/OsrENNl/pCwaMjyIyjHCvVDpnS1GCnmk6XPgblEJwLne+W2PgVBFlYlEuRaV0yVlJPeJh1nsj/UrGW/QsXeePp/Rp5A0IXd/rmbCF0W3d5EAUEcEOT9Fd0CT7hbxGu8O/R9r65g/kbcoF1QlXsB3Ct09+BI7i7BAdGEu+NifaYbyA1o3S+TNg/II8D+rLwA8/3yaHMsDy7FUSFQWL/JtSMxkBZGvGU6ZEjpgFM6eFmUcrGSOV+LMiUYeBE3u9xuJ54F1RUgwdaK1KAlDhVs2iNJijtYiQHScPft8hoyxIzJH8o6ru/85zT0Lnj5Zgwsxbs3Cwu8EKXKoApjdX+uAesw7hXCVVRn1mi7YJHnAXkrdHkdxIL95LFlo0brF0qst5Ta6pLbuiiEURLT1Q0tZ5CQQBA0IgDk9nb4Ngcd7Le4RzzPDOVN2OgiQj1QCa0N8MzoJH/EKAMsyjTOAPwu9mym7Hf26uRkOn2KRsfbuLQmd2xHpEZaKJjo192+bJgtQ2wGa32/LPsfQ7J9Vi/7lgzJQxYGopaa64z3uITDg1MDOLYOf55XiGfzG+nSZSMsv/GL0tqIBSJzcHgHx6MDLf64cE4WlRvK7ozbJRzwmrb/7e2v4/4NmA2uTney6vXzcEsOdPpxU4GOm9lnG2Ybj7KOUZKBDhCW0gKQiuYe9hVNIIwFZIb+bduvboKTUNw3a9bJTQrpBEZxOmkPrwBguQfz41O7iID2PHxl7oa100HI/NDmrTk0vu01oqUJ7p8OmJI+wPPY0BcH0YFDk9SmI+wenAljkjKMf5Mx9uljb/zWrdcLF1qaMCzCxsFBMkizHWOoVmEPq6hwNRad35YsqgQ6Jhr6mRo6GN6cKkJLlzVtXCO+gzzLBKTv9JQ9hr2naIwJHNPjkenRWGuIGs6uCRJD2QirHSuqAw94cYbd4L7EF52yw6kvwu4NKPZgyO/cjsjH7c3pAIGHu6Cx2HHb8hyfprsZ88mk8C82ayhWLnDZxdzNiO6wOn7fAqPHnVzvg0YS6FL12A/PA/8OxrMGVgDncBkTuAhFM2U0v6eRSjLd/G91urO/RJV9DzjyqFb7TgQ1f4E+UPVCbWnwEoB8OmgyEkLRTQQ+khtA34O/UgHzhy/2iOVOtuhOPr0I3PQIQpBtIB0+OzsH7V0iqkpC89VSTUYyOu54jmDTsO6CEh4+WnjPxjs+Lu02RSmk8x7tqCdGLw9XerSVwDnOrnUlObT/COzRDHwNE6inYvQ+toIC+OnsAoYFmHmMzOm1dasqy6DbYNhmaCNyhg9OrPcUuAL7Uy1pyhgNxVW9ypXdMkHvbPgkBAmk9ukQPO2rT/P0RhlqhoM9JZmEnXkXQLpIMbfXIAPmiLEMsNB4XUE3qBx14R/6bH+KJr88UZAzhZT9jB21V6ypPT1HhuHCd3gcsnnP++2U0Kxxp53IQzO4Gzi/3jvWX+6Tw1VCusFNDCYcvn97b7e3NTzoGMxQuJ6k4CMb+z1s3FIYzZavYSwrLWZmYddKLT+K3MopvtZAneZ+gOCcLaE8c45vN5yHAk0POZP/AFBLAwQUAAAACABPcAxdWd7nQrIDAABJCgAAEgAAAHRlc3RzL3Rlc3RfZGF0YS5weZVWS28cKRC+969AnBip3Z6JpZXX0lyiPBRpFe0h2os1QhhoD0k3EKA99kb571vQdA/z2jiWNR6KenxUfVVl1VvjAvrqja5aZ3pkWdh26gGp8eJvOFZVPlimBfMIfq2YZS9B+lCNxt7xRrDAJmtSIfjhTButOOvUv5Jy0w299vXpTetYL0e5kEHyQAetvg+TSb5Rnpsn6WgM42UYpT6wh05Sz3oLf5TI7p/AMejJSZn2TKsW4ML9oqoqIVsU0ecIdLdVcLSMgw9PS3CC7lTYmiFQrzqpk0WnvDKaLO5SrIQerSEzzTuI9iEeyQ+M/mIPssN36B6/ff/508fPeFMjjL6oHuIC3HTznIQfOrNDn94lCcObn4vDFIHv03SR9DlqMu8l5BxgBTJrNjnfC7Reg98RDQTbAygib0o/52pw6rcunXbpy2b+loLm6+Q6JjEzpnFMeenJP6wb5HvnjKtRzwLfrvGcW49zci+xiBTBc6Y3B4XdM4IyBwQZeQI0pt4MDuoMbLJS/H8RZwiYTmatSl5TrZarq9Wba9Zw/xRxHB9vrlarfNzUZ1w5s8ueljVa1WiZtXL9W+V8AEgnFC9L7yU3WlzWgmrZF7I44Ely3MjvA+s8GR2cuddj/Ukq5U2Z2qmXoBY6OAZU8bIDxnhqneqZe4Fsu4GHAfJunQSfT0o/wnNNIKG3NA6auzRfcvLjDTxhukPXCHPFhTD+zXL1J8gcAAmTLylGSqVxsx6NwSIecfySyjCqzF1f6M2yvZem/yaUIxAHGtyvv7ghZ3fWvajRGoeUFvIZPpFj+lGS1W3BXZJQXqMW56L/SNo/m/wovGiCGfiW5HLyrewZ3TK/BciYPfARJdm/JD81DjU/9DHdTZzh4GjnYIjRIJ8DiZJGDL31JYlLCnMz6ADcW90CVYMJrIt89FGyBDriAgnIilOmaI2k5kZAYdd4CO3VLV6cQZrnMHQ0dTIuhtdjDSwMEQ62kZYidpSMw2JGvpwkUet+81u48tgfn/VqTMapR6UhU9l8QnJTdDdsCjuEY40/Co2Lma0vRkovLHfHfuqVQ7xwYB3sSs00nycmZdZK4N44uigsOZs6VLCX6ORkwNWnk2qeT5eSHLs39trxrp7bvv5VZ8MgvM3rLw8XcHd5m5MUsUZFkeSzhVkkRfkePxL9jNKZRN+c1SvL6o8qug/5itKmBRn/xRDpTWNpTxbpXj9MRT+2OWDDrIUPl0ge6VMy76fw6R14c7ymT9SL6KXJPnZV/QdQSwMEFAAAAAgAT3AMXZBFb/MsBgAAexIAAB0AAAB0ZXN0cy90ZXN0X2dyYXBoX3NlcXVlbmNlcy5wea1YbYvjNhD+nl+hGgp263Xzsl1KqAsHx0GhlIPrtxCMNlay6tqyKsl3m9vuf+9oJNuy83L50OwSr6WZR6OZZ0aj3aumJrtGHgmvZaMMKRmT9n22tzOSmqeKP3aTH+F1NvMvoq1BjWoiZDckqShhAH5lOXMIWu2ykhraQWhay4oVu0YYfmibVheSqn9aZoo9r9igc1BUPhWawZTYMd2pxzMCn3cVPwhWfpIVNymOUDtSSMWkakBcs7LQw+xjy6uyBysctmFCN0o7CU0/s6sCn2EF2EcgVDH6TA8snSWD1YMBXBx6rwVWocl6NpuVbE+0YXJlXbHnhzjAFQfztCZcGJKT+xTEFC9ZN7BMyN1vpOQ7s0bDFDOtEuQVX+wnQthoHQy54fECIDAZSS/Iu/WtPP4xEVMgxhVGVLNda7h1ZNMqUFTNFw1qf6l2qmSoOkDIVVtZ3Kii2hSG18yaHk3tcGBMlLIBF8BCVVsLq/YJZ8jvH6cqJQBxQQ1vxDm998P0GeVe4Ynqp0JQsErSHdrZCm7uDGhPdWqOER8Wk03Fd0erU6pGFl+4KJsvUy1HsxK8t7O2oDS+sDIQfZu5b8cZ6qjv2B3jNwYmJeDtgpd6TSquzQaM2KbkoJpWwiiKkH/Jn41gwCH7QBqFieTohBog0mmSRpF99IoLvWU7/Xm9fqya3XM+jxz9IMYgLsvsPWT5BwXuikMuWj1LnSEZ/dYdPEzhn8GUrxA4t+mXXr+6/b1FZA8muRdIiW7b2wCh6AhoS4oDGm0hOifrcEDUAwYyA8/QosU8sz/eIPI9WZEfyeIWwybEG9AWAdryBrS3BB8v4HkhM6qoOLAYsji24UjID2SVktIcJcthel811KyWSaaYfqIyEEzJygEdPZCmStFjvAmMuWDGAA9Me7hPwloUsgoXSslLSo6Jp7BNoAJLssbasjNDnS/gBPHZUgDNaOVqsY5NLQt7Gq3xEErWvq6LsrKEvlbg497/49yJjKJcRCnmS+xc+EuSgFP6mVGBdlt03xRqOVR2t37Wr/uSoYNJnpMYInDf+/eC/HEi74TBpwaJcsicXoFRKUCLVrHH8BX0mJ6CbtYpuVtsz63MygMkhChZaOkyJYvlLUvDbMn3+zgE87GSRoHjLNnaqnJ7Ga1vyVWdIIxSD1hCX7jOF4m1ajFSt7qWtaIpGdZllHm4R3b2g5agHrkfY3psRwNg4hi7tIusxqB+M9iVbsFvLSUdYUfL9zQmPwUHrFfNhPwaJRkDJxgdX9YbrXer0sTamgq+h0hnf2s4eAL9IEWD6ByohNSEUx78VzxCF3BgZfy/JuFmDixMyRKz5ueUPGyToHB6qX7gWueUr7qWCaiU3py0m/nWcmp5TnRE08w0WDGQgpvObqiIm87ybejFnWq0drsdfGOrnDsCfRdZcF2wWppj51XvN3DrcJr60/42P5/62p3BYbl7wHLnx2/0qa9og1vtx2YOwtisGYIa+a7ZNje2rNrOKfHtjDsvsEHOL3fXsd/XKIJObQONLjWtjjBukbSTZTStGoDcYpmbo5Hu1R5kHUbXbUBZNExp14oBaIaikBInhehbkF1vcwUx4Af6qsBI1Qw6RyrBj3AQNq2Bp4JM1c9QZc0TDBSKNapkCupzxxPff30zq1ZbOP9BFpX66whont5OBg45217ysDXYQOG0ZF+6xwoeYaoOEXd6jtSxtSMJxHDrVwVw6eNoaZCxYheWOw5o85P2ZLLyrbJoBBd9IEbm4L2GfKCVhod7sd+heTUz1N5+89e3USVy0eridube6tumfrRvSiZsRBwM7Oakld2OK9Xc8+CGg77DfbGtBIYZgg7q9ssGfETg7v4+3BcxpVThukENT2jjggu/asWFfg4Pq5yMzi3cUuYXcel96bJxpu/fdAC27d9CY7yYz+dX2/+hn7ai4SH0B31kFWK+m0C9JeDqzhGxNd1dx1wl0Pwry5cgS7DzypExXSNhXQVbvv5vkSEjHbaThlbpAUA1Y2V+D6GtueB1W1v/4tUbpmEQeioc7MzR+f3pmWh7Kwfqmq+HuTvaIXTPtrxAYsRe4BzPsKdLMjC1EjSGBu4728Bluq0n5dMD/pqT1ew/UEsDBBQAAAAIAE9wDF2r+kT//gQAADUOAAAbAAAAdGVzdHMvdGVzdF9wcmVwcm9jZXNzaW5nLnB51VZdb9s2FH33ryAEDJA2RZXUdmgNuMC2rkOKAgna7skwCEa6stlKJEHSSbwi++27JCVbStQ0r1MMx+L9Pvfwko2WHamkOhDeKaktqQGUe180TqKY3bX8ahBe4uti0b+IfYdmzBChhiXFRI0L+FH1IngwusoqKRq+HZy0ktU0LJ1UlAalZQXGcHHU/Itxcd6pvQWdkg/AvrItfGINXB6VpV4sFpcfL97/+cdn+vHi4jNZ+SRjShveAqVJpsHI9hriJFNMg7BmXWzQqIaGKM0qyyvW9unEyXJB8OnzXY1Tjb3EPZNwz0gU5CZyv6+YgezAujZKn6R/ysBZtlxMrJNRNutoAlG0WUccC2OWS0EbiVVatwaCXbVQRxvM/h1rDXgXGuxei95TX7xFC8prRIQ3HDSW2e47YSiC1P+mgnVAjdUYzzzA5iF4Xs6MAWzdkHTNLPO5PgiEqzlmuSLR38IFqpckj8YuWNvGHGs1lokK4mCWEswnIVgwCQuEiycFS449H6FIG42R44ScvUHGZm/R/p1bCaVqeWOw0PXGv7mQRrXcphhvL/CfbBoD1iUQx5HVSNYoJWWekjxJSRxds5bXvj+4/DIlRR7WHfBhpcSVHtYhAhc13DqXmomtKxojjVTcg373gHk1yE4bB4Nf+mSSiaYrIGNKgajjbxOJeyJfTbTsq3oo/8CuoEV59FtEeNOn9hMpXdNyAkguEv0ezRg2BVr5NGeFtN5jxIpZeEytRKFQGReNC+5z9GwJQBOcNH1Gp2wCMj+TcsYfcsQRydVbZPmMwrtW3pDztyhvIoT25uybj3l39s2HuZsr9JPc6wrI+aVDqcgz91fMKb7FnnPhyTDVLue0P/MO9VmnnGJePCvKZ2VevCJ5vvSfOZvRJloGYGaUKK2YwlEAtGaH4PysmE2BUuNrC2OU1x6WHpGsMteP2iDvgsn38jBY2wO3y0eA3mq5V/f17+Nwl4yH3Xg/x24nJOPBd28McGuoFO2BenZRJJfTwDF0DYaeNjL1ts5ZbDtF3em49OfNE4fj8dRCle+cZ3EwwDkHUK9elENNZt9a77jXyzBnl60wODY66iEx8ex0S4etPJnQwWUWCr7NzI4p6Cdymc8ojlCYar+c8+pwekRtOgX8vAuGHVjmxjgOcy1xctWj4yI6WoQ2RZuJy+MOf6q7wWDW2zAOiJB21mEDzO8kqWvQU+MwmU+tMkgE0Jmzox27pesfOcv8TojdIE02iUOveJ39sCnoGo+yN26+TdmWGYYXoIGx6f1aJrQ4qoXryYhO2ReDR1mSwS03SLWnWGHoL/IKL5Bjs9Eu3Druma9cGXrD7U7uLe144K6f5cdbR3hD/uOJwMLRWL5ISW0PCla45hF/XvrbnuNc/Colz0OGPFwg0XZ0nYxByWpnVkVKrpitdtTwf2CFHncc+aCRYqs8e53iHUTt2KrAM70FpoVLrBfmefFwTp2eHa/xEkI7xJkjaUGv3Kkz3dTH3Qs1ZtfnOd3XcSh8gvWg6HBzjBZyAM1rIRrWnzbbLBhQvEhVrTTIgFPAlAye77cjuDf3GuH7QzXgRNuCAAQBR9VsczQ7xOsjMuvcle1rR6lgYnNCbT1axnZl+VhWOtlz9/ViKhjW8AZ1FGxmqPDj5peT5pf/i+a7aeRuRQYxG/czyZg49HvyaQxY/3v008fZDKSYFWkr21UBZ78mi/8AUEsDBBQAAAAIAE9wDF3uv6dVkgIAAJoGAAAUAAAAdGVzdHMvdGVzdF9zcGxpdHMucHmVVNuK2zAQffdXDIKCTVOv013SNuBCS+lT/2AJQrHHWYEtq5K8F0L+vSNLsZ2S3VKRh1g6c+Z2ZmSne+NAC1ULC/TTdZI0pu/Amiq3upXOggwgYa08KH4w/aB5eFrBo2hlLRyGCy6Vw4OR7iVJkhobsC/KPaCTVTDDmjdGdJhm8OEr+cp/CCd++pttAnRM/2ShhPvd+NX0Bmw/mAqJt8ZnkAqMUAdMN1nA+xMQZNWwSmg3GLyJRsel8Smv7CObrDw3eeOyXrAWC9pzOLnQGlWdHi9e/GGcRweNbMlLzbYxmNUb2OCUoOHPFagVnT7zNSwmcdoeg8GJXTH5JfbYEpx9YyAbSGNmNzewLuD9RREzeAcfoSyhAGwtAvv+F+EpC61AKqW66JGntVnsrEPrQlMtFwZ5UAc1eI9U26gHy5+ke+iHIAxjsXKyV2ks8ygFatyrIomB2KF1BLuiv3Tup7eYE2l9QXjVt0Onylie+ZUCQsOdEVJ5V2NQZZF/mhFR1nQ/AXjf8IUh4dczPgTl61Oui/naItbl3cf5oqJBC/MinMNOO1vexueQ7TRAlPBrs5WGkuRjzsGMaoM0ohPknlkn3GDZzreaaf9esyXUorvgIQvvhe0yb3FkY5JsBWwuhf/ybWenkUhTKca8KdQlUz5e7l9SFqpCQs4m9lwNSv4ezs2d404nurwTz2k2hrFegqIPg34b3TNsGi+nR5w6FLOdJzUmsYUi39zOTVim5N+KzZfLs4CO+XrQbXF57gLotByHfe8eeIiPRLxUma1QCSP7MCt20AHzf5Pgd9Y5V7+1Ui9ZCu3zYm29OS1hSM7rYjWReZL1inbFCohwvckmuuulvzY/ofbnr38QTDplu1eUmvwBUEsBAhQAFAAAAAgAT3AMXbhswq36BQAAJA0AAAkAAAAAAAAAAAAAAIABAAAAAFJFQURNRS5tZFBLAQIUABQAAAAIAE9wDF0Pr4zJ/gQAAJcMAAAQAAAAAAAAAAAAAACAASEGAABhc3N1bXB0aW9ucy55YW1sUEsBAhQAFAAAAAgAT3AMXc7EHNSTAwAAEwcAABIAAAAAAAAAAAAAAIABTQsAAHBhcGVyX2FsaWdubWVudC5tZFBLAQIUABQAAAAIAE9wDF0AoflHPwAAAD4AAAAQAAAAAAAAAAAAAACAARAPAAByZXF1aXJlbWVudHMudHh0UEsBAhQAFAAAAAgAT3AMXSZNXZPRAgAA+AcAABcAAAAAAAAAAAAAAIABfQ8AAHRyYWNlYWJpbGl0eV9tYXRyaXguY3N2UEsBAhQAFAAAAAgAT3AMXd/0TCMCBAAAFQgAABEAAAAAAAAAAAAAAIABgxIAAGNvbmZpZ3MvYmFzZS55YW1sUEsBAhQAFAAAAAgAT3AMXYgBvYHVAAAAhwEAABsAAAAAAAAAAAAAAIABtBYAAGNvbmZpZ3MvcGFwZXJfZmFpdGhmdWwueWFtbFBLAQIUABQAAAAIAE9wDF1ybqEjmQAAAAkBAAAfAAAAAAAAAAAAAACAAcIXAABjb25maWdzL3ByYWN0aWNhbF9iYXNlbGluZS55YW1sUEsBAhQAFAAAAAgAT3AMXer/t2JFAAAARQAAAA8AAAAAAAAAAAAAAIABmBgAAHNyYy9fX2luaXRfXy5weVBLAQIUABQAAAAIAE9wDF34e5JwIgUAAKIOAAANAAAAAAAAAAAAAACAAQoZAABzcmMvY29uZmlnLnB5UEsBAhQAFAAAAAgAT3AMXbJmmGeyEgAAJEgAAAsAAAAAAAAAAAAAAIABVx4AAHNyYy9kYXRhLnB5UEsBAhQAFAAAAAgAT3AMXWJb6gCIDQAADDIAABYAAAAAAAAAAAAAAIABMjEAAHNyYy9ncmFwaF9zZXF1ZW5jZXMucHlQSwECFAAUAAAACABPcAxdFsEFAuERAAAoSAAAFAAAAAAAAAAAAAAAgAHuPgAAc3JjL3ByZXByb2Nlc3NpbmcucHlQSwECFAAUAAAACABPcAxdtpQP8FgKAABUIgAADQAAAAAAAAAAAAAAgAEBUQAAc3JjL3NwbGl0cy5weVBLAQIUABQAAAAIAE9wDF1UECZANQUAAM4QAAASAAAAAAAAAAAAAACAAYRbAABzcmMvc3RlcDJfc21va2UucHlQSwECFAAUAAAACABPcAxdHP6O5kYHAAC4FwAAEgAAAAAAAAAAAAAAgAHpYAAAc3JjL3N0ZXAzX3Ntb2tlLnB5UEsBAhQAFAAAAAgAT3AMXVne50KyAwAASQoAABIAAAAAAAAAAAAAAIABX2gAAHRlc3RzL3Rlc3RfZGF0YS5weVBLAQIUABQAAAAIAE9wDF2QRW/zLAYAAHsSAAAdAAAAAAAAAAAAAACAAUFsAAB0ZXN0cy90ZXN0X2dyYXBoX3NlcXVlbmNlcy5weVBLAQIUABQAAAAIAE9wDF2r+kT//gQAADUOAAAbAAAAAAAAAAAAAACAAahyAAB0ZXN0cy90ZXN0X3ByZXByb2Nlc3NpbmcucHlQSwECFAAUAAAACABPcAxd7r+nVZICAACaBgAAFAAAAAAAAAAAAAAAgAHfdwAAdGVzdHMvdGVzdF9zcGxpdHMucHlQSwUGAAAAABQAFAAVBQAAo3oAAAAA"

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ARCHIVE_B64))) as project_zip:
    project_zip.extractall(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

if MOUNTED_DATA_DIR.exists():
    DATA_DIR = MOUNTED_DATA_DIR
else:
    DATA_DIR = DOWNLOADED_DATA_DIR
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)
    download_env = os.environ.copy()
    secret_value = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    try:
        classic = json.loads(secret_value)
    except (TypeError, json.JSONDecodeError):
        download_env["KAGGLE_API_TOKEN"] = secret_value
    else:
        download_env["KAGGLE_USERNAME"] = classic["username"]
        download_env["KAGGLE_KEY"] = classic["key"]
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "dungnguyen28101991/cicddos2019-parquet",
         "-p", str(DATA_DIR), "--unzip", "--quiet"],
        env=download_env,
        check=True,
    )
    for key in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        download_env.pop(key, None)
    del secret_value
print(f"Step 3 project ready; using dataset at {DATA_DIR}")


In [ ]:
command = [
    sys.executable, "-m", "src.step3_smoke",
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--config", "configs/base.yaml",
    "--mode-config", "configs/practical_baseline.yaml",
    "--samples-per-file", "2048",
    "--sequence-length", "16",
    "--sequence-stride", "8",
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


In [ ]:
summary_path = OUTPUT_DIR / "step3_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["status"] == "passed", summary
for run in summary["runs"]:
    assert run["row_split_leakage_status"] == "passed", run
    assert run["sequence_leakage_status"] == "passed", run
    assert all(count > 0 for count in run["sequence_counts"].values()), run
summary
